# Process Memory Caps and NumPy Allocation

This notebook uses the helper functions in `draft-exercises.py` to demo a simplified swap idea:
cap this process's memory, try a too-large NumPy allocation, then restore normal behavior.


## 1. Import the helper module and dependencies

Load the helper file directly from the `.py` file.


In [ ]:
from __future__ import annotations

import importlib.util
from pathlib import Path
import sys

import numpy as np
import psutil

helper_path = Path.cwd() / "utils.py"
spec = importlib.util.spec_from_file_location("draft_exercises", helper_path)
assert spec is not None and spec.loader is not None
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)

print(f"Loaded helper from: {helper_path}")
print(f"Python platform: {sys.platform}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/alessandrofelder/dev/slides-large-array-data-osss-2026/draft-exercises.py'

## 2. Inspect system memory and platform

Print RAM size and platform so the demo stays grounded in the current machine.


In [ ]:
vm = psutil.virtual_memory()
print(f"Platform: {sys.platform}")
print(f"Total RAM: {vm.total / 1024**3:.2f} GiB")
print(f"Available RAM: {vm.available / 1024**3:.2f} GiB")
print("Expected helper path:")
print("- POSIX: resource.setrlimit(resource.RLIMIT_AS, ...)")
print("- Windows: Job Object with JOB_OBJECT_LIMIT_PROCESS_MEMORY")

## 3. Apply a process memory cap

Demo: cap this process to 90% of physical RAM.


In [ ]:
cap_bytes = helper.disable_swap(fraction=0.9)
print(f"Applied cap: {cap_bytes:,} bytes")
print(f"Applied cap: {cap_bytes / 1024**3:.2f} GiB")

## 4. Trigger and observe allocation failure

Demo: ask NumPy for an array larger than available RAM and catch the failure.


In [ ]:
target_gib = vm.total / 1024**3 * 2
n_elements = int(target_gib * 1024**3 / np.dtype(np.float64).itemsize)
print(f"Target array size: {target_gib:.2f} GiB")

try:
    arr = np.random.random(n_elements)
    print(f"Unexpected success: {arr.nbytes / 1024**3:.2f} GiB")
except (MemoryError, OSError, ValueError) as exc:
    print(f"Allocation failed as expected: {type(exc).__name__}: {exc}")

## 5. Restore the original memory settings

Restore normal behavior after the demo.


In [ ]:
helper.enable_swap()
print("Memory cap removed.")

## 6. Compare the POSIX and Windows code paths

Show the two implementation paths in the helper: POSIX RLIMIT_AS and Windows Job Objects.


In [2]:
import inspect

source = inspect.getsource(helper)
for line in source.splitlines():
    if "RLIMIT_AS" in line or "JOB_OBJECT_LIMIT_PROCESS_MEMORY" in line or "disable_swap" in line or "enable_swap" in line:
        print(line)

POSIX uses ``resource.setrlimit(RLIMIT_AS, ...)``.
Windows uses a Job Object with ``JOB_OBJECT_LIMIT_PROCESS_MEMORY``.
def disable_swap(fraction: float = 0.9) -> int:
            win32job.JOB_OBJECT_LIMIT_PROCESS_MEMORY
        soft_limit, hard_limit = resource.getrlimit(resource.RLIMIT_AS)
        resource.setrlimit(resource.RLIMIT_AS, (limit_bytes, hard_limit))
def enable_swap() -> None:
    """Remove the process memory cap applied by ``disable_swap()``."""
                win32job.JOB_OBJECT_LIMIT_PROCESS_MEMORY
        _, hard_limit = resource.getrlimit(resource.RLIMIT_AS)
        resource.setrlimit(resource.RLIMIT_AS, (resource.RLIM_INFINITY, hard_limit))
